In [1]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from urllib.parse import urlparse, urlsplit, urlunsplit, quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import re
import os
import requests
import json
import pandas as pd
import time
import random
import cloudscraper
import undetected_chromedriver as uc
import unicodedata
import glob
import csv
from datetime import datetime


In [7]:
def normalizar_texto(texto: str) -> str:
    if not texto:
        return ""
    
    # Garante que caracteres acentuados componham uma única unidade física (ex: "ã" em vez de "a" + "~")
    texto = unicodedata.normalize('NFC', texto)
    
    # Substituição de Espaços não-quebráveis (NBSP) comuns em HTML (\u00a0) e quebras de linha
    texto = texto.replace('\u00a0', ' ').replace("\u2028", " ").replace("\u2029", " ")
    
    # 4. Regex para remoção de caracteres de controle e invisíveis (Fantasmas):
    # \u200b = Zero-Width Space (ZWSP)
    # \u200c = Zero-Width Non-Joiner (ZWNJ)
    # \u200d = Zero-Width Joiner (ZWJ)
    # \ufeff = Byte Order Mark (BOM)
    # \x00-\x1f = Caracteres de controle ASCII não-imprimíveis
    padrao_fantasmas = re.compile(r'[\u200b\u200c\u200d\ufeff]')
    texto = padrao_fantasmas.sub('', texto)
    
    # 5. Normalização de Aspas e Hífens Tipográficos da Web
    # Transforma aspas curvas/inteligentes e travessões longos em caracteres padrão
    substituicoes = {
        '“': '"', '”': '"', '‘': "'", '’': "'",
        '—': '-', '–': '-', '…': '...'
    }
    for caractere_web, caractere_padrao in substituicoes.items():
        texto = texto.replace(caractere_web, caractere_padrao)
        
    # 6. Limpeza e colapso de espaços em branco e quebras de linha duplicadas
    # Transforma sequências de múltiplos espaços, abas ou quebras de linha em um único espaço/quebra
    texto = re.sub(r'[ \t]+', ' ', texto)  # Colapsa múltiplos espaços ou tabs na mesma linha
    texto = re.sub(r'\n\s*\n', '\n', texto)  # Remove linhas puramente em branco consecutivas
    
    return texto.strip()

In [33]:
def corrigir_jsonl(caminho_arquivo='aos_fatos.jsonl', lista_links=None):
    caminho_temp = caminho_arquivo + '.tmp'
    meses = {
        'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04',
        'mai': '05', 'jun': '06', 'jul': '07', 'ago': '08',
        'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
    }
    with open(caminho_arquivo, 'r', encoding='utf-8') as f_in, \
         open(caminho_temp, 'w', encoding='utf-8') as f_out:
        
        for linha in f_in:
            if not linha.strip():
                continue
                
            objeto = json.loads(linha)

            indice = objeto.get('idx')
            objeto['link'] = lista_links[int(indice)]

            texto_original = objeto.pop('text', objeto.get('texto', None))

            if texto_original is not None:
                # Divide o texto pelas quebras de linha
                linhas_texto = texto_original.split('\n')
                
                # Só extrai título e subtítulo se eles ainda não existirem no objeto
                if 'titulo' not in objeto and 'subtitulo' not in objeto:
                    objeto['titulo'] = linhas_texto[0] if len(linhas_texto) > 0 else ""
                    objeto['subtitulo'] = linhas_texto[1] if len(linhas_texto) > 1 else ""
                    objeto['texto'] = "\n".join(linhas_texto[2:]) if len(linhas_texto) > 2 else ""
                else:
                    # Se já tinham sido extraídos, garante que a chave correta é 'texto'
                    objeto['texto'] = texto_original        
                             
            # 2. Renomeia "date" para "data"
            data_original = objeto.pop('date', objeto.get('data', None))
            
            if data_original:
                data_original = data_original.strip()
                
                if '/' in data_original:
                    objeto['data'] = data_original
                else:
                    try:
                        partes = data_original.split()
                        
                        dia = partes[0]   # 1ª posição
                        mes_nome = partes[2].lower() # 3ª posição em lowercase
                        ano = partes[4]   # 5ª posição
                        
                        mes_numero = meses.get(mes_nome[:3], '00')
                        
                        # Monta a data inicial
                        data_formatada = f"{dia}/{mes_numero}/{ano}"
                        
                        # Se o tamanho for menor que 10 (ex: "1/01/2026"), anexa o zero no início
                        if len(data_formatada) < 10:
                            data_formatada = "0" + data_formatada
                            
                        objeto['data'] = data_formatada
                    except:
                        objeto['data'] = data_original
                
            # 3. Renomeia "credits" para "fontes"
            if 'credits' in objeto:
                objeto['fontes'] = objeto['credits']
                del objeto['credits']
                
            # Salva o objeto modificado
            f_out.write(json.dumps(objeto, ensure_ascii=False) + '\n')
            
    os.replace(caminho_temp, caminho_arquivo)
    print(f"Arquivo '{caminho_arquivo}' reestruturado com sucesso!")

In [9]:
def verify(link, fontes) -> bool:
    whitelist = [
        ".aosfatos.org",
        ".agencialupa.org",
        ".gov.br",
        ".estadao.com.br",
        ".uol.com.br",
        ".afp.com"
    ]
    for domain in whitelist:
        if domain in link:
            return True
        if fontes and isinstance(fontes, list):
            for f in fontes:
                if domain in f:
                    return True
    return False

def unificar_bases_jsonl(
    pasta_origem=".",
    arquivo_saida="base_noticias",
):

    dados_consolidados = []

    # Define as colunas obrigatórias na ordem ideal de uma tabela de banco de dados
    colunas_padrao = ["data", "titulo", "subtitulo", "texto", "link", "tags", "fontes", "verificado"]

    # Busca todos os arquivos .jsonl na pasta especificada
    arquivos_jsonl = glob.glob(os.path.join(pasta_origem, "*.jsonl"))

    if not arquivos_jsonl:
        print("Nenhum arquivo .jsonl encontrado na pasta.")
        return

    print(f"Encontrados {len(arquivos_jsonl)} arquivos para unificar...")

    for arquivo in arquivos_jsonl:
        print(f"Lendo: {os.path.basename(arquivo)}")
        with open(arquivo, "r", encoding="utf-8") as f:
            for linha in f:
                if not linha.strip():
                    continue

                objeto = json.loads(linha)

                # Padrão Computacional: Garantir que o registro tenha exatamente as colunas mapeadas
                # Se uma coluna não existir em um determinado arquivo, ela assume um valor padrão (vazio ou lista vazia)
                registro_higienizado = {}
                for coluna in colunas_padrao:
                    if coluna == "verificado":
                        registro_higienizado[coluna] = str(verify(
                            objeto.get("link", ""),
                            objeto.get("fontes", [])
                        ))
                        continue
                    
                    valor = objeto.get(coluna, "")
                    
                    if coluna in ["tags", "fontes"]:
                        # Garante que mesmo os elementos internos da lista sejam strings limpas
                        lista_limpa = (
                            [
                                normalizar_texto(str(item))
                                for item in valor
                                if item
                            ]
                            if isinstance(valor, list)
                            else []
                        )
                        registro_higienizado[coluna] = lista_limpa
                    else:
                        texto_limpo = (
                            normalizar_texto(str(valor).strip()) if valor else ""
                        )
                        registro_higienizado[coluna] = texto_limpo

                dados_consolidados.append(registro_higienizado)

    # Cria o DataFrame do Pandas
    df = pd.DataFrame(dados_consolidados)

    # Garante a ordem exata das colunas na estrutura da tabela
    df = df[colunas_padrao]
    
    if not (df['data'].astype(str).str.len() == 10).all():
        print("Atenção: Nem todas as datas estão no formato 'dd/mm/yyyy'. Verifique os dados para garantir a consistência temporal.")
        return

    # Tratamento computacional de tipos no Pandas
    # Força a coluna 'data' para o tipo datetime do Pandas (opcional, mas altamente recomendado)
    # df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y', errors='coerce')

    print(
        f"\nTotal de registros unificados: {len(df)} linhas e {len(df.columns)} colunas."
    )

    nome_final = f"{arquivo_saida}.csv"

    # Antes de salvar em CSV, convertemos as listas em strings separadas por '|'
    # para não quebrar a estrutura de colunas do CSV tradicional
    df_csv = df.copy()
    for col in ["tags", "fontes"]:
        df_csv[col] = df_csv[col].apply(
            lambda x: "|".join(x) if isinstance(x, list) else ""
        )

    # Salva usando separador ponto e vírgula (padrão Excel BR) e codificação utf-8-sig
    df_csv.to_csv(
        nome_final, sep=";", index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL
    )
    print(f"Sucesso! Base salva em formato texto: {nome_final}")

    return df


# --- COMO EXECUTAR ---
# Supondo que seus arquivos estão na mesma pasta do script:
df_noticias = unificar_bases_jsonl()


Encontrados 6 arquivos para unificar...
Lendo: agencia_lupa.jsonl
Lendo: projeto_comprova.jsonl
Lendo: afp_checamos.jsonl
Lendo: aos_fatos.jsonl
Lendo: boatos_org.jsonl
Lendo: saude_com_ciencia.jsonl

Total de registros unificados: 4648 linhas e 8 colunas.
Sucesso! Base salva em formato texto: base_noticias.csv


In [6]:
def ordenar_jsonl_por_data(caminho_entrada, caminho_saida):
    """
    Abre um arquivo JSONL, ordena suas linhas por data (formato dd/mm/aaaa)
    e salva o resultado em um novo arquivo JSONL.
    """
    objetos = []
    
    # 1. Ler o arquivo JSONL linha por linha
    with open(caminho_entrada, 'r', encoding='utf-8') as file:
        for linha in file:
            if linha.strip():  # Ignora linhas vazias
                objetos.append(json.loads(linha))
                
    # 2. Ordenar a lista de objetos baseado no atributo "data"
    # Convertemos a string 'dd/mm/aaaa' em um objeto datetime real para ordenar corretamente
    objetos.sort(
        key=lambda x: datetime.strptime(x['data'], '%d/%m/%Y') if x.get('data') else datetime.min,
        reverse=True
    )
    
    # 3. Salvar os objetos ordenados de volta em um arquivo JSONL
    with open(caminho_saida, 'w', encoding='utf-8') as file:
        for obj in objetos:
            file.write(json.dumps(obj, ensure_ascii=False) + '\n')
            
    print(f"✓ Arquivo ordenado salvo com sucesso em: {caminho_saida}")
    return objetos

noticias_ordenadas = ordenar_jsonl_por_data('aos_fatos.jsonl', 'aos_fatos.jsonl')

# Passo 2: Extrair os links mantendo a ordenação cronológica
links_lista = []
for noticia in noticias_ordenadas:
    # Verificamos se o atributo 'link' existe e não está vazio
    if noticia.get('link'):
        links_lista.append(noticia['link'])

# Passo 3: Salvar a lista pura de links em um arquivo .json tradicional
with open('aosfatos.json', 'w', encoding='utf-8') as file:
    json.dump(links_lista, file, ensure_ascii=False, indent=4)

print(f"✓ Lista com {len(links_lista)} links extraídos e salva em: aosfatos.json")

✓ Arquivo ordenado salvo com sucesso em: aos_fatos.jsonl
✓ Lista com 700 links extraídos e salva em: aosfatos.json


### Saúde com Ciência (SUS)

In [ ]:
def get_page(url_page):
    try:
        html = urlopen(url_page)
    except HTTPError:
        # Se a página retornar um erro HTTP (como 404), interrompe
        return []
    bs = BeautifulSoup(html, 'html.parser')
    links_noticias = []
    
    seletor = 'main article .tileHeadline a'
    for anchor in bs.select(seletor):
        if anchor.has_attr('href'):
            links_noticias.append(anchor['href'])
            
    return links_noticias


def get_noticias(base_url):
    links_noticias = []
    contador = 0
    
    while True:
        url_page = f"{base_url}?b_start:int={contador}"
        print(f"Coletando notícias de: {url_page}")
        
        links_pagina = get_page(url_page)
        
        if not links_pagina:
            print(f"Nenhuma notícia encontrada em {contador}. Finalizando paginação.")
            break
            
        links_noticias.extend(links_pagina)
        
        contador += 10
        
    return links_noticias

links_noticias = get_noticias("https://www.gov.br/saude/pt-br/assuntos/saude-com-ciencia/noticias")

with open('saude_com_ciencia.json', 'w', encoding='utf-8') as file:
    json.dump(links_noticias, file, ensure_ascii=False, indent=4)
    
print("Lista salva com sucesso usando JSON!")

In [ ]:
with open('saude_com_ciencia.json', 'r', encoding='utf-8') as file:
    links = json.load(file)
    
def extrair_noticias(links, pos=0):
    links = links[pos:]    

    with open('saude_com_ciencia.jsonl', 'a', encoding='utf-8') as file:
        for i, url in enumerate(links):
            idx = i + pos
            
            try:
                html = urlopen(url)
                bs = BeautifulSoup(html, 'html.parser')
                
                main_tag = bs.find('main')
                article_tag = main_tag and main_tag.find('article') 
                
                
                doc_modified = main_tag and (main_tag.find(class_='documentModified') or main_tag.find(class_='documentPublished'))
                
                val_tag = doc_modified and doc_modified.find(class_='value') 
                
                heading_el = article_tag and article_tag.find(class_='documentFirstHeading')
                desc_el = article_tag and article_tag.find(class_='documentDescription')
                core_el = article_tag and article_tag.find(id='content-core')
                
                all_categories = main_tag.select('a.link-category') if main_tag else []
                tags = [a.get_text(strip=True) for a in all_categories if not a.find_parent('article')]
                
                if not (main_tag and val_tag and article_tag and heading_el and desc_el and core_el):
                    print(f"[Break] Elemento ausente na posição [{idx}]: {url}")
                    break
                
                date_text = val_tag.get_text().lstrip()
                date = date_text.split(' ')[0] if ' ' in date_text else date_text
                
                def limpar_texto(elemento):
                    if not elemento:
                        return ""
                    for img in elemento.find_all('img'): 
                        img.decompose()
                    for a in elemento.find_all('a'):
                        a.replace_with(f"{a.get_text(strip=True)}")

                    for inline in elemento.find_all(['strong', 'em', 'span', 'b', 'i', 'u']):
                        inline.unwrap()
                    
                    blocos_internos = elemento.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li'])
                    if blocos_internos:
                        linhas_finais = []
                        for bloco in blocos_internos:
                            # Pega o texto do parágrafo/heading, remove QUALQUER \n de dentro dele e limpa espaços duplos
                            texto_bloco = " ".join(bloco.get_text().replace('\n', ' ').split())
                            
                            if texto_bloco:
                                linhas_finais.append(texto_bloco)
                        
                        return "\n".join(linhas_finais)       
                    else:
                        # Caso o elemento seja simples, limpa tudo e transforma em uma linha contínua
                        texto_bloco = elemento.get_text().replace('\n', ' ')
                        return " ".join(texto_bloco.split())
                
                texto_final = f"{limpar_texto(heading_el)}\n{limpar_texto(desc_el)}\n{limpar_texto(core_el)}"
                
                dados = {
                    'idx': idx,
                    'date': date,
                    'tags': tags,
                    'text': texto_final
                }
                
                file.write(json.dumps(dados, ensure_ascii=False) + '\n')
                
            except Exception as e:
                print(f"Erro crítico na posição [{idx}] ({url}): {e}")
                break

    print(f"\nLote processado! Os dados foram salvos.")
    
extrair_noticias(links)

### Aos Fatos (Saúde)

In [ ]:
def get_page(url_page):
    try:
        # Criamos um cabeçalho simulando o navegador Google Chrome
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
        
        # Passamos a URL e os headers para o Request
        req = Request(url_page, headers=headers)
        html = urlopen(req)
        
    except HTTPError as e:
        # Agora você verá exatamente qual erro HTTP o site está devolvendo (ex: 403, 503)
        print(f"[ERRO HTTP] Falha ao acessar {url_page}. Código: {e.code} - {e.reason}")
        return []
        
    bs = BeautifulSoup(html, 'html.parser')
    links_noticias = []
    
    anchors = bs.select('.grid a')
    for anchor in anchors:
        if anchor.has_attr('href') and "/noticias/" in anchor['href'] and "?tag" not in anchor['href'] and "?formato" not in anchor['href'] and "?canal" not in anchor['href']:
            links_noticias.append(anchor['href'])
            
    return links_noticias

# tags: "saude", "oms", "ministerio-da-saude", "covid-19"
def get_noticias(base_url):
    with open('aosfatos.json', 'r', encoding='utf-8') as file:
        links = json.load(file)
    links_noticias = set(links)  # Usamos um set para evitar duplicatas
    contador = 1
    
    while contador <= 35: 
        url_page = f"{base_url}" + (f"&page={contador}" if contador > 1 else "")
        print(f"Coletando notícias de: {url_page}")
        
        links_pagina = get_page(url_page)
        
        if not links_pagina:
            print(f"Erro em {contador}. Finalizando paginação.")
            break
            
        links_noticias.update(links_pagina)
        
        tempo_espera = random.uniform(1.0, 3.0) 
        time.sleep(tempo_espera)
        
        contador += 1
        
    return list(links_noticias)

links_noticias = get_noticias("https://www.aosfatos.org/noticias/?tag=covid-19")

with open('aosfatos.json', 'w', encoding='utf-8') as file:
    json.dump(links_noticias, file, ensure_ascii=False, indent=4)
    
print("Lista salva com sucesso usando JSON!")

In [ ]:
def filtrar_links(caminho_arquivo='aosfatos.json'):
    # 1. Carrega a lista do arquivo JSON
    with open(caminho_arquivo, 'r', encoding='utf-8') as file:
        links = json.load(file)
    
    # 2. Filtra removendo as strings que contêm "?tag" ou "?formato"
    links_filtrados = [
        "https://www.aosfatos.org"+link for link in links 
    ]
    
    return links_filtrados
dados_limpos = filtrar_links('aosfatos.json')
with open('aosfatos.json', 'w', encoding='utf-8') as file:
    json.dump(dados_limpos, file, ensure_ascii=False, indent=4)

In [ ]:
with open('aosfatos.json', 'r', encoding='utf-8') as file:
    links = json.load(file)
    
def extrair_noticias(links, pos=0):
    links = links[pos:]    

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    with open('aos_fatos.jsonl', 'a', encoding='utf-8') as file:
        for i, url in enumerate(links):
            idx = i + pos
            print(f"Processando [{idx}]: {url}")
            
            try:
                req = Request(url, headers=headers)
                html = urlopen(req)
                bs = BeautifulSoup(html, 'html.parser')
                
                article_tag = bs.select_one('.prose') 
                
                time_tag = article_tag and article_tag.find('aside') 
                article_parent = article_tag and article_tag.parent
                tags = article_parent and [a.get_text(strip=True) for a in article_parent.select('a[href*="?tag"]')]
                tags = list(set(tags)) if tags else []

                if not (time_tag and article_tag and tags):
                    print(f"[Break] Elemento ausente na posição [{idx}]: {url}")
                    break
                
                date = re.search(r".*?\d{4}", time_tag.get_text(strip=True)).group()
                
                def limpar_texto(elemento):
                    if not elemento:
                        return ""
                    for img in elemento.find_all('img'): 
                        img.decompose()
                    for a in elemento.find_all('a'):
                        a.replace_with(f"{a.get_text(strip=True)}")

                    for inline in elemento.find_all(['strong', 'em', 'span', 'b', 'i', 'u']):
                        inline.unwrap()
                    
                    blocos_internos = elemento.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li'])
                    if blocos_internos:
                        linhas_finais = []
                        for bloco in blocos_internos:
                            # Pega o texto do parágrafo/heading, remove QUALQUER \n de dentro dele e limpa espaços duplos
                            texto_bloco = " ".join(bloco.get_text().replace('\n', ' ').split())
                            
                            if texto_bloco:
                                linhas_finais.append(texto_bloco)
                        
                        return "\n".join(linhas_finais)       
                    else:
                        # Caso o elemento seja simples, limpa tudo e transforma em uma linha contínua
                        texto_bloco = elemento.get_text().replace('\n', ' ')
                        return " ".join(texto_bloco.split())
                
                texto_final = f"{limpar_texto(article_tag)}"
                
                dados = {
                    'idx': idx,
                    'date': date,
                    'tags': tags,
                    'text': texto_final
                }
                
                file.write(json.dumps(dados, ensure_ascii=False) + '\n')
                
            except Exception as e:
                print(f"Erro na posição [{idx}] ({url}): {e}")
                break
            
            time.sleep(random.uniform(1.5, 3.5))

    print(f"\nLote processado! Os dados foram salvos.")
    
extrair_noticias(links, pos=0)

In [38]:
# Remover a substring "\nCompartilhe\nAdicione o Aos Fatos às suas fontes do Google"

with open('saude_com_ciencia.json', 'r', encoding='utf-8') as file:
    links = json.load(file)
corrigir_jsonl('saude_com_ciencia.jsonl', links)

Arquivo 'saude_com_ciencia.jsonl' reestruturado com sucesso!


### Boatos.org (Saúde)

In [ ]:
def get_page(url_page):
    try:
        # Criamos um cabeçalho simulando o navegador Google Chrome
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
        
        # Passamos a URL e os headers para o Request
        req = Request(url_page, headers=headers)
        html = urlopen(req)
        
    except HTTPError as e:
        # Agora você verá exatamente qual erro HTTP o site está devolvendo (ex: 403, 503)
        print(f"[ERRO HTTP] Falha ao acessar {url_page}. Código: {e.code} - {e.reason}")
        return []
        
    bs = BeautifulSoup(html, 'html.parser')
    links_noticias = []
    
    articles = bs.select('main article')
    
    for idx, article in enumerate(articles, start=1):
        anchors = article.select('.entry-title a')
        
        if len(anchors) > 1:
            print(f"[ERRO] O artigo #{idx} na página {url_page} contém múltiplos ({len(anchors)}) elementos '.entry-title a'.")
            return False
        elif len(anchors) == 1 and anchors[0].has_attr('href') and anchors[0]['href'].startswith('http'):
            links_noticias.append(anchors[0]['href'])
        else:
            # Se qualquer uma das condições acima falhar, disparamos o aviso
            print(f"[AVISO] O artigo #{idx} na página {url_page} está sem um '.entry-title' ou possui um href inválido.")
            return False
            
    return links_noticias


def get_noticias(base_url):
    links_noticias = []
    contador = 1
    
    while contador <= 52: 
        url_page = f"{base_url}" + (f"/page/{contador}" if contador > 1 else "")
        print(f"Coletando notícias de: {url_page}")
        
        links_pagina = get_page(url_page)
        
        if not links_pagina:
            print(f"Erro em {contador}. Finalizando paginação.")
            break
            
        links_noticias.extend(links_pagina)
        
        tempo_espera = random.uniform(1.0, 3.0) 
        time.sleep(tempo_espera)
        
        contador += 1
        
    return links_noticias

links_noticias = get_noticias("https://www.boatos.org/saude")

with open('boatos_org.json', 'w', encoding='utf-8') as file:
    json.dump(links_noticias, file, ensure_ascii=False, indent=4)
    
print("Lista salva com sucesso usando JSON!")

In [ ]:
with open('boatos_org.json', 'r', encoding='utf-8') as file:
    links = json.load(file)
    
def extrair_noticias(links, pos=0):
    links = links[pos:]    

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    with open('boatos_org.jsonl', 'a', encoding='utf-8') as file:
        for i, url in enumerate(links):
            idx = i + pos
            print(f"Processando [{idx}]: {url}")
            
            try:
                req = Request(url, headers=headers)
                html = urlopen(req)
                bs = BeautifulSoup(html, 'html.parser')
                
                main_tag = bs.find('main')
                article_tag = main_tag and main_tag.find('article') 
                
                time_tag = main_tag and (main_tag.find('time', class_='updated') or main_tag.find('time', class_='published'))

                heading_el = article_tag and article_tag.find(class_='entry-title')

                content_el = article_tag and article_tag.find(class_='entry-content')

                if not (main_tag and time_tag and article_tag and heading_el and content_el):
                    print(f"[Break] Elemento ausente na posição [{idx}]: {url}")
                    break
                
                date_text = time_tag.get_text().lstrip()
                date = date_text.split(' ')[0] if ' ' in date_text else date_text
                
                def limpar_texto(elemento):
                    if not elemento:
                        return ""
                    for img in elemento.find_all('img'): 
                        img.decompose()
                    for a in elemento.find_all('a'):
                        a.replace_with(f"{a.get_text(strip=True)}")

                    for inline in elemento.find_all(['strong', 'em', 'span', 'b', 'i', 'u']):
                        inline.unwrap()
                    
                    blocos_internos = elemento.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li'])
                    if blocos_internos:
                        linhas_finais = []
                        for bloco in blocos_internos:
                            # Pega o texto do parágrafo/heading, remove QUALQUER \n de dentro dele e limpa espaços duplos
                            texto_bloco = " ".join(bloco.get_text().replace('\n', ' ').split())
                            
                            if texto_bloco:
                                linhas_finais.append(texto_bloco)
                        
                        return "\n".join(linhas_finais)       
                    else:
                        # Caso o elemento seja simples, limpa tudo e transforma em uma linha contínua
                        texto_bloco = elemento.get_text().replace('\n', ' ')
                        return " ".join(texto_bloco.split())
                
                texto_final = f"{limpar_texto(heading_el)}\n{limpar_texto(content_el)}"
                
                dados = {
                    'idx': idx,
                    'date': date,
                    'text': texto_final
                }
                
                file.write(json.dumps(dados, ensure_ascii=False) + '\n')
                
            except Exception as e:
                print(f"Erro na posição [{idx}] ({url}): {e}")
                break
            
            time.sleep(random.uniform(1.5, 3.5))

    print(f"\nLote processado! Os dados foram salvos.")
    
extrair_noticias(links, pos=1119)

In [ ]:
with open('boatos_org.json', 'r', encoding='utf-8') as file:
    links = json.load(file)
    
def extrair_links(links, pos=0):
    links = links[pos:]    

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    with open('boatos_org_links.jsonl', 'a', encoding='utf-8') as file:
        for i, url in enumerate(links):
            idx = i + pos
            print(f"Processando [{idx}]: {url}")
            
            try:
                req = Request(url, headers=headers)
                html = urlopen(req)
                bs = BeautifulSoup(html, 'html.parser')
                
                anchors = bs.select('article a')
                
                hrefs_validos = [
                    a['href'] for a in anchors 
                    if a.has_attr('href') and a['href'].startswith('http')
                ]
                
                dados = {
                    'idx': idx,
                    'links': hrefs_validos
                }
                
                file.write(json.dumps(dados, ensure_ascii=False) + '\n')
                
            except Exception as e:
                print(f"Erro na posição [{idx}] ({url}): {e}")
                break
            
            time.sleep(random.uniform(1.5, 3.5))

    print(f"\nLote processado! Os dados foram salvos.")
    
extrair_links(links, pos=0)

In [ ]:
def is_url_segura(url, whitelist):
    if not url.startswith(('http://', 'https://')):
        if not url.startswith('//'):
            url = '//' + url
            
    netloc = urlparse(url).hostname
    if not netloc:
        return False
    netloc = netloc.lower()

    for sufixo in whitelist:
        sufixo = sufixo.lower()
        
        if netloc == sufixo:
            return True
            
        if netloc.endswith(sufixo):
            # Descobre onde o sufixo começa dentro do netloc
            pos = netloc.rfind(sufixo)
            caractere_anterior = netloc[pos - 1]
            
            if caractere_anterior == '.':
                return True
    return False

def limpar_links():
    new_links = []
    
    whitelist = [
        "projetocomprova.com.br",
        "checamos.afp.com",
        "aosfatos.org",
        
        "doctoralia.com.br",
        
        "gov.br", 
        "leg.br", 
        "jus.br", 
        "mp.br", 
        "def.br", 
        "mil.br", 
        "rio",
        
        "globo.com",
        "abril.com.br",
        "acidadeon.com",
        "acordacidade.com.br",
        "an.com.br",
        "atarde.com.br",
        "atribuna.com.br",
        "bnews.com.br",
        "boqnews.com",
        "brasil61.com",
        "correio24horas.com.br",
        "correiobraziliense.com.br",
        "correiodecarajas.com.br",
        "correiodopovo.com.br",
        "correiope.com.br",
        "costanorte.com.br",
        "diarinho.com.br",
        "diariocatarinense.com.br",
        "diariocomercial.com.br",
        "diariodagente.app",
        "diariodaregiao.com.br",
        "diariodenoticias.com.br",
        "diariodepernambuco.com.br",
        "diariodoacionista.com.br",
        "diariodoamazonas.com.br",
        "diariodocomercio.com.br",
        "diariodolitoral.com.br",
        "diariodonordeste.com.br",
        "diariodopara.com.br",
        "diariodorio.com",
        "em.com.br",
        "emdiaes.com.br",
        "entreriosjornal.com",
        "es360.com.br",
        "eshoje.com.br",
        "estadao.com.br",
        "extra.inf.br",
        "folha.com.br",
        "folha1.com.br",
        "folhabv.com.br",
        "folhadelondrina.com.br",
        "folhape.com.br",
        "folhavitoria.com.br",
        "g1.com.br",
        "gauchazh.com.br",
        "gaz.com.br",
        "gazetadelimeira.com.br",
        "gazetadigital.com.br",
        "gazetadopovo.com.br",
        "gazetaonline.com.br",
        "gazetasp.com.br",
        "grupoahora.net.br",
        "hojeemdia.com.br",
        "ilustrado.com.br",
        "jc.com.br",
        "jcam.com.br",
        "jcnet.com.br",
        "jornalcidade.net",
        "jornalcidade.net.br",
        "jornalcruzeiro.com.br",
        "jornaldaboaesperanca.com.br",
        "jornaldaorla.com.br",
        "jornaldebrasilia.com.br",
        "jornaldezminutos.com",
        "jornaldocomercio.com",
        "jornaldopovo.com.br",
        "jornaldotocantins.com.br",
        "jornaldr1.com.br",
        "jornalexemplo.com.br",
        "jornalnh.com.br",
        "lance.com.br",
        "liberal.com.br",
        "meionorte.com",
        "moneytimes.com.br",
        "monitormercantil.com.br",
        "ndmais.com.br",
        "nfnoticias.com.br",
        "oestadoonline.com.br",
        "oglobo.com.br",
        "oliberal.com.br",
        "ootimista.com.br",
        "opopular.com.br",
        "opovo.com.br",
        "orm.com.br",
        "otempo.com.br",
        "pioneiro.clicrbs.com.br",
        "portalarauto.com.br",
        "portalbenews.com.br",
        "portalcorreio.com.br",
        "portaltemponovo.com.br",
        "propmark.com.br",
        "rac.com.br",
        "romanews.com.br",
        "santa.com.br",
        "seucreditodigital.com.br",
        "seudinheiro.com",
        "tnonline.com.br",
        "todahora.com",
        "tribunadosertao.com.br",
        "tribunadosmunicipios.com.br",
        "tribunahoje.com",
        "tribunaonline.com.br",
        "uol.com.br",
        "valor.globo.com",
        "vozdosmunicipios.com.br"
    ]
    with open('boatos_org_links.jsonl', 'r', encoding='utf-8') as file:
        for linha in file:
            if not linha.strip():
                continue 
                
            dados = json.loads(linha)
            
            dados['links'] = [
                url for url in dados['links']
                #if ".pdf" not in url.lower()
                if is_url_segura(url, whitelist) 
            ]
            
            new_links.append(json.dumps(dados, ensure_ascii=False) + '\n')
            
    with open('boatos_org_links.jsonl', 'w', encoding='utf-8') as file:
        file.writelines(new_links)

    print(f"Arquivo 'boatos_org_links.jsonl' atualizado com sucesso!")

limpar_links()

### Lupa (Saúde)

In [ ]:
### COM SELENIUM
# COLOCAR NUM SET E ABRIR PÁGINAS DE TAGS ESPECÍFICAS: "covid-19", "pandemia", "vacinacao", "saude", "cancer", "dengue", "anvisa", "oms", "sus" e "ministerio-da-saude"

def get_page(driver, url_page):
    try:
        driver.get(url_page)
        time.sleep(random.uniform(4.0, 6.0)) # Tempo padrão para o carregamento inicial
        
        # Primeira tentativa de leitura
        html_da_pagina = driver.page_source
        bs = BeautifulSoup(html_da_pagina, 'html.parser')
        articles = bs.select('.archive-list-feed article')
        
        # --- LOGICA DE INTERVENÇÃO HUMANA ---
        # Se não achou os artigos OU se detectou assinaturas do Cloudflare/Bloqueio
        html_lower = html_da_pagina.lower()
        if not articles or "cloudflare" in html_lower or "captcha" in html_lower:
            print(f"\n🛑 [⚠️ CAPTCHA OU BLOQUEIO DETECTADO] na página: {url_page}")
            print("👉 Vá até a janela do Chrome e resolva a verificação humana manualmente.")
            
            # O Python vai congelar aqui e esperar você interagir com o terminal
            input("👉 APÓS resolver o desafio e a página carregar as notícias, volte aqui e pressione [ENTER]: ")
            
            print("🔄 Retomando raspagem... Capturando HTML atualizado pós-captcha.")
            # Captura o HTML novamente, agora com o CAPTCHA resolvido
            html_da_pagina = driver.page_source
            bs = BeautifulSoup(html_da_pagina, 'html.parser')
            articles = bs.select('.archive-list-feed article')
        # -------------------------------------
            
    except Exception as e:
        print(f"[ERRO] Falha ao acessar {url_page}: {e}")
        return []
        
    links_noticias = []
    
    # Se mesmo após o seu ENTER ainda não tiver artigos, o script avisa
    if not articles:
        print(f"[AVISO] Nenhum artigo encontrado na página {url_page} mesmo após a pausa.")
        return []
    
    for idx, article in enumerate(articles, start=1):
        # tag_element = article.select_one('.feed-hat')
        # if not tag_element:
        #     continue
            
        # tag = tag_element.get_text(strip=True).lower()
        # if "saúde" not in tag and "saude" not in tag:
        #     continue
            
        anchor = article.select_one('a.feed-link')
        
        if anchor and anchor.has_attr('href') and anchor['href'].startswith('http') and "latamchequea" not in anchor['href'] and "/verificamos-latam" not in anchor['href']:
            links_noticias.append(anchor['href'])
        else:
            print(f"[AVISO] O artigo #{idx} na página {url_page} possui um href inválido.")
            continue 
            
    return links_noticias


def get_noticias(base_url, arquivo_destino='agencialupa.json'):
    try:
        with open(arquivo_destino, 'r', encoding='utf-8') as file:
            links_noticias = set(json.load(file))
    except (FileNotFoundError, json.JSONDecodeError):
        links_noticias = set()
        
    print(f"Total de links coletados até agora: {len(links_noticias)}")
    contador = 1  # Começando da página onde você parou
    
    options = uc.ChromeOptions()
    options.add_argument('--start-maximized')
    options.add_argument('--disable-gpu')
    
    print("Iniciando o navegador...")
    driver = uc.Chrome(options=options, version_main=140)
    
    try:
        while contador <= 13: 
            url_page = f"{base_url}" + (f"/page/{contador}" if contador > 1 else "")
            print(f"\n--------------------------------------------------")
            print(f"Coletando notícias de: {url_page}")
            
            links_pagina = get_page(driver, url_page)
            
            # Se retornou links com sucesso (ou lista vazia legítima pós-intervenção)
            if isinstance(links_pagina, list) and links_pagina:
                links_noticias.update(set(links_pagina))
                
                # Salva o arquivo JSON atualizado
                with open(arquivo_destino, 'w', encoding='utf-8') as file:
                    json.dump(list(links_noticias), file, ensure_ascii=False, indent=4)
                
                print(f"-> Sucesso! {len(links_pagina)} links identificados.")
            
            # Delay normal entre as páginas humanas
            print(f"Total acumulado: {len(links_noticias)} links.")
            tempo_espera = random.uniform(5.0, 15.0) 
            time.sleep(tempo_espera)
            
            contador += 1
            
    finally:
        print("\nFechando o navegador...")
        driver.quit()
        
    return links_noticias


# Execução
links_noticias = get_noticias("https://www.agencialupa.org/tag/ministerio-da-saude")

with open('agencialupa.json', 'w', encoding='utf-8') as file:
    json.dump(list(links_noticias), file, ensure_ascii=False, indent=4)

print("\nProcesso finalizado!")

In [ ]:
# Carrega os links do arquivo de configuração
with open('agencialupa.json', 'r', encoding='utf-8') as file:
    links = json.load(file)
    
def extrair_noticias(links, pos=0):
    links = links[pos:]    

    options = uc.ChromeOptions()
    # options.add_argument('--headless') # Descomente se quiser rodar em segundo plano
    options.add_argument('--start-maximized')
    options.add_argument('--disable-gpu')
    
    print("Iniciando o navegador Chrome controlado pelo Selenium...")
    
    # NOTA: Se persistir o erro de "Binary Location", adicione o parâmetro:
    # browser_executable_path="/usr/bin/google-chrome"
    driver = uc.Chrome(options=options, version_main=140)
    
    with open('agencia_lupa.jsonl', 'a', encoding='utf-8') as file:
        for i, url in enumerate(links):
            idx = i + pos
            print(f"Processando [{idx}]: {url}")
            
            try:
                driver.get(url)
                
                # Tempo para simular leitura humana e carregar a página (e permitir sua supervisão)
                time.sleep(random.uniform(4.0, 7.0))
                
                html_da_pagina = driver.page_source
                html_lower = html_da_pagina.lower()
                bs = BeautifulSoup(html_da_pagina, 'html.parser')

                # =========================================================================
                # 🛡️ --- ADICIONADO: LÓGICA DE INTERVENÇÃO HUMANA ANTI-BOT ---
                # Se não achar o cabeçalho da notícia OU se detectar assinaturas do Cloudflare/Captcha
                if not bs.select_one('section.single-header') or "cloudflare" in html_lower or "captcha" in html_lower:
                    print(f"\n🛑 [⚠️ CAPTCHA OU BLOQUEIO DETECTADO] na notícia [{idx}]: {url}")
                    print("👉 Vá até a janela do Chrome e resolva a verificação humana manualmente.")
                    
                    # O Python vai congelar aqui e esperar você interagir com o terminal
                    input("👉 APÓS resolver o desafio e a página exibir a notícia completa, volte aqui e pressione [ENTER]: ")
                    
                    print("🔄 Retomando raspagem... Capturando HTML atualizado pós-captcha.")
                    # Recaptura o HTML e o Soup atualizados após a sua liberação
                    html_da_pagina = driver.page_source
                    html_lower = html_da_pagina.lower()
                    bs = BeautifulSoup(html_da_pagina, 'html.parser')
                # =========================================================================
                
                # 1. Seleciona a seção do cabeçalho da notícia
                header_section = bs.select_one('section.single-header')
                if not header_section:
                    print(f"[Aviso] Seção 'single-header' ausente na posição [{idx}]: {url}")
                    break
                
                # 2. Extração do Título
                title_tag = header_section.select_one('.single-head-title')
                
                # 3. Extração da Data com Lógica de Prioridade (Atualizado > Publicado)
                time_tags = header_section.select('time')
                target_time_tag = None
                
                # Procura primeiro pela tag que diz "Atualizado"
                for t in time_tags:
                    if t.has_attr('title') and "Atualizado" in t['title']:
                        target_time_tag = t
                        break
                
                # Se não achou, procura pela tag que diz "Publicado"
                if not target_time_tag:
                    for t in time_tags:
                        if t.has_attr('title') and "Publicado" in t['title']:
                            target_time_tag = t
                            break
                
                # Se achou a tag de tempo, extrai a data por Regex (dd/mm/aaaa)
                date = ""
                if target_time_tag and target_time_tag.has_attr('title'):
                    texto_data = target_time_tag['title']
                    match_data = re.search(r"\d{2}/\d{2}/\d{4}", texto_data)
                    if match_data:
                        date = match_data.group()
                
                # 4. Extração do Conteúdo Principal e das Tags
                article_tag = bs.select_one('article')
                content_tag = article_tag.select_one('.single-content')
                
                # Captura todas as tags baseado na classe 'tax-links-item'
                tag_elements = article_tag.select('a.tax-links-item')
                lista_tags = [t['title'].strip() for t in tag_elements if t.has_attr('title')]
                
                # Validação estrita de elementos essenciais
                if not (title_tag and content_tag and date and target_time_tag and lista_tags):
                    print(f"[Aviso] Elemento essencial ausente (Título, Conteúdo ou Data) na posição [{idx}]: {url}")
                    break
                
                # Função de limpeza de HTML fornecida por você
                def limpar_texto(elemento):
                    if not elemento:
                        return ""
                    for img in elemento.find_all('img'): 
                        img.decompose()
                    for a in elemento.find_all('a'):
                        a.replace_with(f"{a.get_text(strip=True)}")

                    for inline in elemento.find_all(['strong', 'em', 'span', 'b', 'i', 'u']):
                        inline.unwrap()
                    
                    blocos_internos = elemento.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li'])
                    if blocos_internos:
                        linhas_finais = []
                        for bloco in blocos_internos:
                            texto_bloco = " ".join(bloco.get_text().replace('\n', ' ').split())
                            if texto_bloco:
                                linhas_finais.append(texto_bloco)
                        return "\n".join(linhas_finais)       
                    else:
                        texto_bloco = elemento.get_text().replace('\n', ' ')
                        return " ".join(texto_bloco.split())
                
                # Processa os textos finais purificados
                titulo_limpo = limpar_texto(title_tag)
                texto_limpo = limpar_texto(content_tag)
                
                # Formata as tags como string separada por | para o seu padrão do Pandas
                tags_formatadas = lista_tags
                
                # Monta o dicionário estruturado para o seu TCC
                dados = {
                    'idx': idx,
                    'link': url,
                    'data': date,
                    'titulo': titulo_limpo,
                    'subtitulo': texto_limpo.split('\n', 1)[0] if texto_limpo else "",
                    'tags': tags_formatadas,
                    'texto': texto_limpo.split('\n', 1)[1] if texto_limpo else ""
                }
                
                # Escreve imediatamente no arquivo JSONL
                file.write(json.dumps(dados, ensure_ascii=False) + '\n')
                print(f"✓ Salvo com sucesso! Data: {date} | Tags: {len(lista_tags)}")
                
            except Exception as e:
                print(f"❌ Erro na posição [{idx}] ({url}): {e}")
                # Mudado para continue para o script não fechar o navegador por um erro isolado
                continue
            
            # Delay entre requisições de notícias
            time.sleep(random.uniform(2.0, 5.0))
        
    print("\nFechando o navegador...")
    driver.quit()
    print(f"\nLote processado! Os dados foram salvos.")
    
# Execução iniciando da posição desejada
extrair_noticias(links, pos=292)

### Projeto Comprova (Saúde)

In [ ]:
def get_page(url_page):
    try:
        # Criamos um cabeçalho simulando o navegador Google Chrome
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
        
        # Passamos a URL e os headers para o Request
        req = Request(url_page, headers=headers)
        html = urlopen(req)
        
    except HTTPError as e:
        # Agora você verá exatamente qual erro HTTP o site está devolvendo (ex: 403, 503)
        print(f"[ERRO HTTP] Falha ao acessar {url_page}. Código: {e.code} - {e.reason}")
        return "block"
        
    bs = BeautifulSoup(html, 'html.parser')
    links_noticias = []
    
    articles = bs.select('main article.answer')
    
    for idx, article in enumerate(articles, start=1):
        anchor = article.select_one('a.answer__title__link')
        
        if anchor and anchor.has_attr('href') and anchor['href'].startswith('http'):
            links_noticias.append(anchor['href'])
        else:
            # Se qualquer uma das condições acima falhar, disparamos o aviso
            print(f"[AVISO] O artigo #{idx} na página {url_page} possui um href inválido.")
            return False
            
    return links_noticias


def get_noticias(base_url):
    try:
        with open('projeto_comprova.json', 'r', encoding='utf-8') as file:
            links_noticias = json.load(file)
        print(f"Total de links coletados até agora: {len(links_noticias)}")
        contador = 1
        
        while contador <= 47: 
            url_page = f"{base_url}" + (f"/page/{contador}" if contador > 1 else "") + "/?filter=saude"
            print(f"Coletando notícias de: {url_page}")
            
            links_pagina = get_page(url_page)
            
            if links_pagina is False:
                print(f"Erro em {contador}. Finalizando paginação.")
                break
            
            if links_pagina == "block":
                print("🚨 Tomamos um 429 (Too Many Requests)! Dormindo por 5 minutos...")
                with open('projeto_comprova.json', 'w', encoding='utf-8') as file:
                    json.dump(links_noticias, file, ensure_ascii=False, indent=4)
                time.sleep(300) # 5 minutos de soneca para o IP esfriar
                continue
                
            links_noticias.extend(links_pagina)
            
            tempo_espera = random.uniform(5.0, 15.0) 
            time.sleep(tempo_espera)
            
            contador += 1
            print(f"Total de links coletados até agora: {len(links_noticias)}")
    except:
        with open('projeto_comprova.json', 'w', encoding='utf-8') as file:
            json.dump(links_noticias, file, ensure_ascii=False, indent=4)
        print("Ocorreu um erro ao coletar as notícias.")
            
    return links_noticias

links_noticias = get_noticias("https://projetocomprova.com.br")

with open('projeto_comprova.json', 'w', encoding='utf-8') as file:
    json.dump(links_noticias, file, ensure_ascii=False, indent=4)
    
print("Lista salva com sucesso usando JSON!")

In [ ]:
with open('projeto_comprova.json', 'r', encoding='utf-8') as file:
    links = json.load(file)
    
def extrair_noticias(links, pos=0):
    links = links[pos:]    

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    with open('projeto_comprova.jsonl', 'a', encoding='utf-8') as file:
        for i, url in enumerate(links):
            idx = i + pos
            print(f"Processando [{idx}]: {url}")
            
            try:
                url_parts = list(urlsplit(url))
                # Aplica o quote apenas no path (componente de índice 2) para codificar o "publicações"
                url_parts[2] = quote(url_parts[2])
                url_formatada = urlunsplit(url_parts)
                
                req = Request(url_formatada, headers=headers)
                html = urlopen(req)
                
                bs = BeautifulSoup(html, 'html.parser')
                
                article_tag = bs.select_one('main article') 
                title_tag = article_tag and article_tag.select_one('.answer__title__link')
                subtitle_tag1 = article_tag and article_tag.select_one('.answer__tag')
                subtitle_tag2 = article_tag and article_tag.select_one('.answer__tag__details')
                credits_tags = article_tag.select('.answer__credits__link') if article_tag else []
                time_tag = article_tag and article_tag.select_one('.answer__credits__date') 

                if not (time_tag and article_tag and title_tag and subtitle_tag1 and subtitle_tag2 and credits_tags):
                    print(f"[Break] Elemento ausente na posição [{idx}]: {url}")
                    break
                
                date = time_tag.get_text(strip=True)
                
                def limpar_texto(elemento):
                    if not elemento:
                        return ""
                    for img in elemento.find_all('img'): 
                        img.decompose()
                    for a in elemento.find_all('a'):
                        a.replace_with(f"{a.get_text(strip=True)}")

                    for inline in elemento.find_all(['strong', 'em', 'span', 'b', 'i', 'u']):
                        inline.unwrap()
                    
                    blocos_internos = elemento.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li'])
                    if blocos_internos:
                        linhas_finais = []
                        for bloco in blocos_internos:
                            # Pega o texto do parágrafo/heading, remove QUALQUER \n de dentro dele e limpa espaços duplos
                            texto_bloco = " ".join(bloco.get_text().replace('\n', ' ').split())
                            
                            if texto_bloco:
                                linhas_finais.append(texto_bloco)
                        
                        return "\n".join(linhas_finais)       
                    else:
                        # Caso o elemento seja simples, limpa tudo e transforma em uma linha contínua
                        texto_bloco = elemento.get_text().replace('\n', ' ')
                        return " ".join(texto_bloco.split())
                
                texto_final = f"{limpar_texto(title_tag)}\n{limpar_texto(subtitle_tag1)}: {limpar_texto(subtitle_tag2)}\n{limpar_texto(article_tag)}"
                
                dados = {
                    'idx': idx,
                    'date': date,
                    'credits': [credit.get('href') for credit in credits_tags if credit.has_attr('href')],
                    'text': texto_final
                }
                
                file.write(json.dumps(dados, ensure_ascii=False) + '\n')
                
            except Exception as e:
                print(f"Erro na posição [{idx}] ({url}): {e}")
                break
            
            time.sleep(random.uniform(1.5, 3.5))

    print(f"\nLote processado! Os dados foram salvos.")
    
extrair_noticias(links, pos=252)

### AFP Checamos (Saúde)

In [ ]:
# 1. Inicializa o navegador
driver = webdriver.Chrome()

# 2. Abre o site desejado
driver.get("https://checamos.afp.com/list/topics/146")

print("====================================================")
print("VEJA A TELA DO NAVEGADOR:")
print("1. Clique no botão 'Carregar mais' quantas vezes quiser.")
print("2. Quando terminar de carregar tudo, volte aqui no terminal.")
print("====================================================")

# O script vai parar aqui e esperar você interagir com a página
input("Pressione [ENTER] aqui no terminal para iniciar o scraping do conteúdo...")

print("Extraindo os links (hrefs)...")

articles = driver.find_elements(By.CSS_SELECTOR, "main article.node")

links_coletados = []

for article in articles:
    try:
        # Busca a PRIMEIRA (find_element no singular) tag 'a' dentro DESTE article específico
        primeira_ancora = article.find_element(By.TAG_NAME, "a")
        
        # Extrai o atributo 'href' (o link)
        href = primeira_ancora.get_attribute("href")
        
        if href and href.startswith("http"):
            links_coletados.append(href)
            print(f"Link encontrado: {href}")
        else:
            print(f"[AVISO] O artigo no índice {articles.index(article)} possui uma tag 'a' sem um href válido.")
            
    except Exception:
        # Se um article por acaso não tiver nenhuma tag 'a', ele ignora e vai para o próximo
        continue

print("\n--- Resultado final ---")
print(f"Total de links coletados: {len(links_coletados)}")
with open('afp_checamos.json', 'w', encoding='utf-8') as file:
    json.dump(links_coletados, file, ensure_ascii=False, indent=4)

# 4. Fecha o navegador ao terminar
driver.quit()
print("Scraping finalizado com sucesso!")

In [ ]:
with open('afp_checamos.json', 'r', encoding='utf-8') as file:
    links = json.load(file)
    
def extrair_noticias(links, pos=0):
    links = links[pos:]    

    options = uc.ChromeOptions()
    # options.add_argument('--headless') # Descomente esta linha se quiser que o navegador rode em segundo plano (escondido)
    options.add_argument('--start-maximized')
    options.add_argument('--disable-gpu')
    
    print("Iniciando o navegador Chrome controlado pelo Selenium...")
    # Inicializa o navegador com as correções antibot
    driver = uc.Chrome(options=options, version_main=140)
    
    with open('afp_checamos.jsonl', 'a', encoding='utf-8') as file:
        for i, url in enumerate(links):
            idx = i + pos
            print(f"Processando [{idx}]: {url}")
            
            try:
                # O Selenium abre a página
                driver.get(url)
                
                # Aguarda um tempo aleatório para simular a leitura humana e dar tempo da página carregar
                time.sleep(random.uniform(3.0, 6.0))
                
                # Pegamos o HTML que já foi renderizado pelo navegador
                html_da_pagina = driver.page_source
                
                # O BeautifulSoup entra em ação para parsear o HTML coletado
                bs = BeautifulSoup(html_da_pagina, 'html.parser')
                
                article_tag = bs.select_one('main article.node') 
                title_tag = article_tag and article_tag.select_one('.sub-header').find()
                subtitle_tag = article_tag and article_tag.select_one('.wrapper-summary')
                content_tag = article_tag and article_tag.select_one('.wrapper-body')
                time_tag = article_tag and article_tag.select_one('.date-full-format') 

                if not (time_tag and article_tag and title_tag and subtitle_tag and content_tag):
                    print(f"[Break] Elemento ausente na posição [{idx}]: {url}")
                    break
                
                date = re.search(r"\d.*?\d{4}", time_tag.get_text(strip=True)).group()

                def limpar_texto(elemento):
                    if not elemento:
                        return ""
                    for img in elemento.find_all('img'): 
                        img.decompose()
                    for a in elemento.find_all('a'):
                        a.replace_with(f"{a.get_text(strip=True)}")

                    for inline in elemento.find_all(['strong', 'em', 'span', 'b', 'i', 'u']):
                        inline.unwrap()
                    
                    blocos_internos = elemento.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li'])
                    if blocos_internos:
                        linhas_finais = []
                        for bloco in blocos_internos:
                            # Pega o texto do parágrafo/heading, remove QUALQUER \n de dentro dele e limpa espaços duplos
                            texto_bloco = " ".join(bloco.get_text().replace('\n', ' ').split())
                            
                            if texto_bloco:
                                linhas_finais.append(texto_bloco)
                        
                        return "\n".join(linhas_finais)       
                    else:
                        # Caso o elemento seja simples, limpa tudo e transforma em uma linha contínua
                        texto_bloco = elemento.get_text().replace('\n', ' ')
                        return " ".join(texto_bloco.split())
                
                texto_final = f"{limpar_texto(title_tag)}\n{limpar_texto(subtitle_tag)}\n{limpar_texto(content_tag)}"
                
                dados = {
                    'idx': idx,
                    'date': date,
                    'text': texto_final
                }
                
                file.write(json.dumps(dados, ensure_ascii=False) + '\n')
                
            except Exception as e:
                print(f"Erro na posição [{idx}] ({url}): {e}")
                break
            
            time.sleep(random.uniform(2.0, 4.0))
        
    print("\nFechando o navegador...")
    driver.quit()
    print(f"\nLote processado! Os dados foram salvos.")
    
extrair_noticias(links, pos=353)